In [1]:
!uv pip install chardet # Used to detect encoding of a web page

Resolved 1 package in 80ms                                           
Installed 1 package in 27ms                                 
 + chardet==5.2.0


In [2]:
!uv pip install requests

Audited 1 package in 4ms


In [ ]:
from pathlib import Path

BASE_PATH = Path.cwd().joinpath("data")
BASE_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
import requests

probable_encoding = "windows-1252"

url_codigo_minero = "https://servicios.infoleg.gob.ar/infolegInternet/anexos/40000-44999/43797/texact.htm"
url_codigo_penal = "https://servicios.infoleg.gob.ar/infolegInternet/anexos/15000-19999/16546/texact.htm"

CODIGO_MINERO_FILE_PATH = BASE_PATH.joinpath("codigo_minero_43797.html")
CODIGO_PENAL_FILE_PATH = BASE_PATH.joinpath("codigo_penal_16546.html")

response = requests.get(url_codigo_minero, verify=False)
response.encoding = probable_encoding  # Set the correct encoding


/home/leospairani/.uv-envs/tesis-notebooks/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'servicios.infoleg.gob.ar'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
# Ensure the request was successful
if response.status_code == 200:
    # Save the content to a file ignoring UnicodeEncodeError errors.
    with open(CODIGO_MINERO_FILE_PATH, "w", encoding=probable_encoding) as file:
        file.write(response.text)
else:
    print(f"Failed to retrieve the webpage. Status code: {response.status_code}")


In [79]:
from bs4 import BeautifulSoup
import re

response.encoding = "windows-1252"  # Set the correct encoding
soup = BeautifulSoup(response.text, "html.parser")

In [58]:
from pprint import pprint

In [ ]:
previous_tag_type = None

norm_data = {
    "norm_identifier": None,
    "norm_type": None,
    "norm_title": None,
    "norm_number": None,
    "sanction_norm_type": None,
    "sanction_norm_number": None,
    "sanction_formula": None,
    "sanction_date": None,
    "promulgation_date": None,
    "publication_date": None,
    "articles": [],
    "annexes": [],
}

article = {
    "parent_node": None,
    "article_number": None,
    "article_text": None,
}

annex = {
    "parent_node": None,
    "annex_number": None,
    "annex_title": None,
    "annex_type": None,
    "annex_content": None,
}

parent_node = None

patterns = {
    # "norm_type": r"ley\s+n°\s*\d+",
    # "norm_title": r"código\s+minero",
    "sanction_norm_type_ley": r"ley\s+n°\s*\d+",
    "sanction_formula": r"sancionan con fuerza de",
    "sanction_date": [
        r"a\s+(\d{1,2})\s+de\s+(\w+)\s+de\s+(\d{4})", # "25 de noviembre de 1886"
    ],
    "article_number": [
            r"^Art\. (\d+)° –", # "Art. 7° – Sin perjuicio de lo dispuesto en los"
            r"^Art\. (\d+)\. –", # "Art. 34. – Sin perjuicio ..."
        ],
    # "norm_number": r"n°\s*\d+",
    # "norm_date": r"\d{1,2}\s+de\s+\w+\s+de\s+\d{4}",
}

"""
If the norm is a code:
- the norm_type is "codigo"
- the norm_title is "código <name>"
- the sanction_norm_type is "ley" if it contains a law number
- the sanction_norm_number is the number of the law that sanctions the code
"""


for tag in soup.find("body"):

    # Ignore blank lines
    text = tag.text.strip("\n")
    if text == "":
        continue
    # Normalize text
    text = text.replace("\n", " ").replace("\r", "").strip()
    if not previous_tag_type:
        # Extract norm type and title
        if not norm_data["norm_type"]:
            if "codigo" in text.lower():
                norm_data["norm_type"] = "codigo"
                if not norm_data["norm_title"]:
                    norm_data["norm_title"] = text.lower()
            if  norm_data["norm_title"]:
                previous_tag_type = "norm_type"
                continue
    if norm_data["norm_type"] == "codigo":
        # Extract sanction norm type
        if not norm_data["sanction_norm_type"]:
            match = re.search(patterns["sanction_norm_type_ley"], text, re.IGNORECASE)
            if match:
                norm_data["sanction_norm_type"] = "ley"
                norm_data["sanction_norm_number"] = match.group(0).split(" ")[-1]
                previous_tag_type = "sanction_norm_type"
                continue
    # Extract sanction formula
    if not norm_data["sanction_formula"] and (previous_tag_type == "sanction_norm_type" or previous_tag_type == "norm_type"):
        match = re.search(patterns["sanction_formula"], text, re.IGNORECASE)
        if match:
            text = text.replace(":", "").strip()
            norm_data["sanction_formula"] = text
            previous_tag_type = "sanction_formula"
            continue
    # Extract sanction date
    if not norm_data["sanction_date"]:
        for pattern in patterns["sanction_date"]:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                norm_data["sanction_date"] = " ".join(match.groups())
                previous_tag_type = "sanction_date"
                break
        if norm_data["sanction_date"]:
            continue

    # Check for annexes or data after the last article from the main body of the norm
    if previous_tag_type == "sanction_date" or "anexo" in text.lower():
        # annex = {
        #     "parent_node": None,
        #     "annex_number": None,
        #     "annex_title": None,
        #     "annex_type": None,
        #     "annex_text": None,
        # }
        # Check if tag has the align="center" attribute
        if tag.name == "p" and tag.get("align") == "center":
            # Extract annex number and title
            if "codigo" in text.lower():
                annex["annex_type"] = "codigo"
                annex["annex_title"] = text.lower()
                annex["parent_node"] = parent_node
                norm_data["annexes"].append(annex.copy())
                previous_tag_type = "annex"
                parent_node = annex["annex_title"]
                annex = {
                    "parent_node": None,
                    "annex_number": None,
                    "annex_title": None,
                    "annex_type": None,
                    "annex_content": None,
                }
                continue

    # Extract articles
    if "artículo" in text.lower() or "art." in text.lower():
        for pattern in patterns["article_number"]:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                text = text.replace(match.group(0), "").strip()
                article["article_number"] = match.group(1)
                article["article_text"] = text
                norm_data["articles"].append(article.copy())
                previous_tag_type = "article"
                article = {
                    "parent_node": parent_node,
                    "article_number": None,
                    "article_text": None,
                }
                break
        # continue

    # Set norm identifier
    if not norm_data["norm_identifier"]:
        if norm_data["norm_type"] == "codigo" and norm_data["sanction_norm_type"] and norm_data["sanction_norm_number"]:
            norm_data["norm_identifier"] = f"{norm_data['sanction_norm_type']} {norm_data['sanction_norm_number']}"
            parent_node = norm_data["norm_identifier"]
        elif norm_data["norm_type"] and norm_data["norm_number"]:
            norm_data["norm_identifier"] = f"{norm_data['norm_type']} {norm_data['norm_number']}"
            parent_node = norm_data["norm_identifier"]


    print(text)
# print(norm_data)
pprint(norm_data)


Sanción Ley 1919
Título I DE LAS MINAS Y SU DOMINIO  arts. 1 a 20    Título II DE LAS PERSONAS QUE PUEDEN ADQUIRIR MINAS  arts. 21 a 24    Título III DE LAS RELACIONES ENTRE EL PROPIETARIO Y EL  MINERO  arts. 25 a 43    Título IV DE LA ADQUISICION DE LAS MINAS  arts. 44 a 71    Título V DE LAS PERTENENCIAS Y SU DEMARCACION  arts. 72 a 98    Título VI DE LOS EFECTOS DE LA CONCESION DE LAS PERTENENCIAS  arts. 99 a 108    Título VII DE LAS OTRAS ADQUISICIONES QUE REQUIEREN CONCESION  arts. 109 a 145    Título VIII DE LA EXPLOTACION  arts. 146 a 170    Título IX DISPOSICIONES ESPECIALES SOBRE LAS SUSTANCIAS DE LA SEGUNDA CATEGORIA  arts. 171 a 200    Título X  DISPOSICIONES CONCERNIENTES A LAS SUSTANCIAS DE LA TERCERA CATEGORIA  arts. 201 a 204    Título XI DE LOS MINERALES NUCLEARES  arts. 205 a 212    Título XII DE LAS CONDICIONES DE LA CONCESION  arts. 213 a 232     Título XIII CONDICIONES DE LA EXPLOTACION  arts. 233 a 268    Título XIV DE LOS AVIOS DE MINAS  arts. 269 a 285    Título

'       \n\n\n\n\nCODIGO DE MINERIA\n\nINDICE TEMATICO\n\n\n\n\nSanción Ley 1919\n\n\n\n\nTítulo\nI\nDE LAS MINAS Y SU DOMINIO\n\narts.\n1 a 20 \n\n\nTítulo\nII\nDE LAS PERSONAS QUE PUEDEN ADQUIRIR\nMINAS\n\narts.\n21 a 24 \n\n\nTítulo\nIII\nDE LAS RELACIONES ENTRE EL\nPROPIETARIO Y EL\xa0\nMINERO\n\narts.\n25 a 43 \n\n\nTítulo\nIV\nDE LA ADQUISICION DE LAS MINAS\n\narts.\n44 a 71 \n\n\nTítulo\nV\nDE LAS PERTENENCIAS Y SU DEMARCACION\n\narts.\n72 a 98 \n\n\nTítulo VI\nDE LOS EFECTOS DE LA CONCESION DE LAS PERTENENCIAS\n\narts. 99 a 108\n\n\n\nTítulo\nVII\nDE LAS OTRAS ADQUISICIONES QUE REQUIEREN CONCESION\n\narts. 109 a 145\n\n\n\nTítulo\nVIII\nDE LA EXPLOTACION\n\narts. 146 a 170\n\n\n\nTítulo IX\nDISPOSICIONES ESPECIALES SOBRE LAS SUSTANCIAS DE LA\nSEGUNDA\nCATEGORIA\n\narts. 171 a 200\n\n\n\nTítulo X\n\nDISPOSICIONES CONCERNIENTES A LAS SUSTANCIAS DE LA\nTERCERA\nCATEGORIA\n\narts. 201 a 204\n\n\n\nTítulo\nXI\nDE LOS MINERALES NUCLEARES\n\narts. 205 a 212\n\n\n\nTítulo\nXII\nDE LAS 

In [ ]:
# Obtener Código de Minería
BASE_PATH = Path.cwd()
BASE_FILE_PATH = BASE_PATH.joinpath("codigo_minero_43797.html")
ENCODING = "ISO-8859-1"

source = "http://servicios.infoleg.gob.ar/infolegInternet/anexos/40000-44999/43797/texact.htm"  # document per local path or URL


with open(BASE_FILE_PATH.with_suffix(".md"), "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

with open(BASE_FILE_PATH.with_suffix(".html"), "w", encoding="utf-8") as f:
    f.write(result.document.export_to_html())

In [ ]:
# Obtener Código Penal
BASE_PATH = Path.cwd()
BASE_FILE_PATH = BASE_PATH.joinpath("codigo_penal_16546.html")
ENCODING = "ISO-8859-1"

source = "http://servicios.infoleg.gob.ar/infolegInternet/anexos/15000-19999/16546/texact.htm"  # document per local path or URL


with open(BASE_FILE_PATH.with_suffix(".md"), "w", encoding="utf-8") as f:
    f.write(result.document.export_to_markdown())

with open(BASE_FILE_PATH.with_suffix(".html"), "w", encoding="utf-8") as f:
    f.write(result.document.export_to_html())